In [13]:
from langgraph.graph import StateGraph ,START ,END
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
from dotenv import load_dotenv
load_dotenv()
from typing import TypedDict
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-72B-Instruct",
    task = "text generation"
)
model = ChatHuggingFace(llm=llm)

In [2]:
class BlogState(TypedDict):
    title:str
    outline:str
    content:str
    

In [3]:
graph = StateGraph(BlogState)

In [12]:
def create_outline(state:BlogState)-> BlogState:
    title = state['title']
    prompt = f"Create an outline for a blog post with the title: {title}"
    outline = model.invoke(prompt).content
    state['outline'] = outline
    return state

In [5]:
def create_blog(state:BlogState)-> BlogState:
    outline = state['outline']
    title = state['title']
    prompt = f"Create a blog post based on the following outline: {outline},and the following title: {title}"
    content = model.invoke(prompt).content
    state['content'] = content
    return state

In [ ]:
    # nodes:
# nodes:
if "create_outline" not in graph.nodes:
    graph.add_node("create_outline", create_outline)
graph.add_node("create_blog", create_blog)
#edges:
graph.add_edge(START, "create_outline")
graph.add_edge("create_outline", "create_blog")
graph.add_edge("create_blog", END)

Adding a node to a graph that has already been compiled. This will not be reflected in the compiled graph.


ValueError: Node `create_outline` already present.

In [8]:
workflow = graph.compile()

In [10]:
intial_state = {"title": "Rise of AI in India"}
final_state = workflow.invoke(intial_state)
print(final_state['outline'])


ValueError: You must provide an api_key to work with auto API or log in with `hf auth login`.